In [7]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=5000)
client.admin.command("ping")   

db = client["kopi_nusantara"]
menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


In [8]:

menu.delete_many({})

menu.insert_many([
    
    {"nama_menu": "Espresso",          "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 100},
    {"nama_menu": "Americano",         "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 90},
    {"nama_menu": "Kopi Tubruk",       "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 80, "rekomendasi_barista": True},
    {"nama_menu": "Cappuccino",        "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 75},
    {"nama_menu": "Kopi Susu Gula Aren","harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 120, "rekomendasi_barista": True},
    {"nama_menu": "Caffe Latte",       "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 70},
    {"nama_menu": "V60 Single Origin", "harga": 35000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 40, "rekomendasi_barista": True},

    {"nama_menu": "Nasi Goreng Kampung","harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 50},
    {"nama_menu": "Mie Goreng Pedas",  "harga": 24000, "kategori": "Makanan", "pedas_level": 3, "stok": 45},
    {"nama_menu": "Roti Bakar Coklat", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 60},
    {"nama_menu": "Kentang Goreng",    "harga": 20000, "kategori": "Makanan", "pedas_level": 1, "stok": 65},
    {"nama_menu": "Pisang Goreng",     "harga": 13000, "kategori": "Makanan", "pedas_level": 0, "stok": 55},


    {"nama_menu": "Es Teh Lemon",      "harga": 10000, "kategori": "Non-Kopi", "stok": 150},
    {"nama_menu": "Coklat Panas",      "harga": 20000, "kategori": "Non-Kopi", "stok": 85},
])

print("Jumlah dokumen di collection 'menu':", menu.count_documents({}))
print("Per kategori:")
for k in menu.aggregate([{"$group": {"_id": "$kategori", "jumlah": {"$sum": 1}}}, {"$sort": {"_id": 1}}]):
    print(f"  - {k['_id']}: {k['jumlah']}")

Jumlah dokumen di collection 'menu': 14
Per kategori:
  - Kopi: 7
  - Makanan: 5
  - Non-Kopi: 2


In [3]:
print("Menu Kopi dengan tingkat_kepahitan > 3:")
for m in menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}, {"_id": 0}):
    print(f"  - {m['nama_menu']:<20} harga=Rp{m['harga']:,}  pahit={m['tingkat_kepahitan']}")

Menu Kopi dengan tingkat_kepahitan > 3:
  - Espresso             harga=Rp15,000  pahit=5
  - Americano            harga=Rp18,000  pahit=4
  - Kopi Tubruk          harga=Rp12,000  pahit=5
  - V60 Single Origin    harga=Rp35,000  pahit=4


In [4]:
print("Menu dengan harga antara Rp15.000 - Rp30.000:")
for m in menu.find({"harga": {"$gte": 15000, "$lte": 30000}}, {"_id": 0, "nama_menu": 1, "kategori": 1, "harga": 1}).sort("harga", 1):
    print(f"  - {m['nama_menu']:<20} ({m['kategori']}) Rp{m['harga']:,}")

Menu dengan harga antara Rp15.000 - Rp30.000:
  - Espresso             (Kopi) Rp15,000
  - Roti Bakar Coklat    (Makanan) Rp15,000
  - Americano            (Kopi) Rp18,000
  - Kentang Goreng       (Makanan) Rp20,000
  - Coklat Panas         (Non-Kopi) Rp20,000
  - Kopi Susu Gula Aren  (Kopi) Rp22,000
  - Mie Goreng Pedas     (Makanan) Rp24,000
  - Cappuccino           (Kopi) Rp25,000
  - Caffe Latte          (Kopi) Rp27,000
  - Nasi Goreng Kampung  (Makanan) Rp28,000


In [5]:
print("Menu TANPA field pedas_level:")
for m in menu.find({"pedas_level": {"$exists": False}}, {"_id": 0, "nama_menu": 1, "kategori": 1}):
    print(f"  - {m['nama_menu']:<20} ({m['kategori']})")

Menu TANPA field pedas_level:
  - Espresso             (Kopi)
  - Americano            (Kopi)
  - Kopi Tubruk          (Kopi)
  - Cappuccino           (Kopi)
  - Kopi Susu Gula Aren  (Kopi)
  - Caffe Latte          (Kopi)
  - V60 Single Origin    (Kopi)
  - Es Teh Lemon         (Non-Kopi)
  - Coklat Panas         (Non-Kopi)


In [9]:
# 1) Semua dokumen -> Tersedia
r = menu.update_many({}, {"$set": {"status": "Tersedia"}})
print(f"update_many : cocok={r.matched_count}, diubah={r.modified_count}")

# 2) Dua menu pilihan -> Habis (satu update_one per menu)
r1 = menu.update_one({"nama_menu": "Americano"},           {"$set": {"status": "Habis"}})
r2 = menu.update_one({"nama_menu": "Nasi Goreng Kampung"}, {"$set": {"status": "Habis"}})
print(f"update_one  : Americano diubah={r1.modified_count}, Nasi Goreng Kampung diubah={r2.modified_count}")

print("\nStatus seluruh menu:")
for m in menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {m['nama_menu']:<22} {m['status']}")

update_many : cocok=14, diubah=14
update_one  : Americano diubah=1, Nasi Goreng Kampung diubah=1

Status seluruh menu:
  - Espresso               Tersedia
  - Americano              Habis
  - Kopi Tubruk            Tersedia
  - Cappuccino             Tersedia
  - Kopi Susu Gula Aren    Tersedia
  - Caffe Latte            Tersedia
  - V60 Single Origin      Tersedia
  - Nasi Goreng Kampung    Habis
  - Mie Goreng Pedas       Tersedia
  - Roti Bakar Coklat      Tersedia
  - Kentang Goreng         Tersedia
  - Pisang Goreng          Tersedia
  - Es Teh Lemon           Tersedia
  - Coklat Panas           Tersedia


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, BooleanType)

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


dokumen = list(menu.find({}))
print("Jumlah dokumen diambil dari MongoDB:", len(dokumen))


skema = StructType([
    StructField("_id",                 StringType(),  False),
    StructField("nama_menu",           StringType(),  False),
    StructField("kategori",            StringType(),  False),
    StructField("harga",               IntegerType(), False),
    StructField("tingkat_kepahitan",   IntegerType(), True),   
    StructField("pedas_level",         IntegerType(), True),   
    StructField("rekomendasi_barista", BooleanType(), True),   
    StructField("stok",                IntegerType(), True),
    StructField("status",              StringType(),  True),
])


baris = [
    (
        str(d["_id"]),                      
        d["nama_menu"],
        d["kategori"],
        d["harga"],
        d.get("tingkat_kepahitan"),
        d.get("pedas_level"),
        d.get("rekomendasi_barista"),
        d.get("stok"),
        d.get("status"),
    )
    for d in dokumen
]

df_menu = spark.createDataFrame(baris, schema=skema)
print("Konversi ke Spark DataFrame berhasil.")

Jumlah dokumen diambil dari MongoDB: 14
Konversi ke Spark DataFrame berhasil.


In [12]:
df_menu.printSchema()

# (pelengkap) bukti bahwa field yang tidak ada terisi null
df_menu.select("nama_menu", "kategori", "tingkat_kepahitan", "pedas_level", "status").show(truncate=False)

root
 |-- _id: string (nullable = false)
 |-- nama_menu: string (nullable = false)
 |-- kategori: string (nullable = false)
 |-- harga: integer (nullable = false)
 |-- tingkat_kepahitan: integer (nullable = true)
 |-- pedas_level: integer (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- stok: integer (nullable = true)
 |-- status: string (nullable = true)



+-------------------+--------+-----------------+-----------+--------+
|nama_menu          |kategori|tingkat_kepahitan|pedas_level|status  |
+-------------------+--------+-----------------+-----------+--------+
|Espresso           |Kopi    |5                |NULL       |Tersedia|
|Americano          |Kopi    |4                |NULL       |Habis   |
|Kopi Tubruk        |Kopi    |5                |NULL       |Tersedia|
|Cappuccino         |Kopi    |3                |NULL       |Tersedia|
|Kopi Susu Gula Aren|Kopi    |2                |NULL       |Tersedia|
|Caffe Latte        |Kopi    |2                |NULL       |Tersedia|
|V60 Single Origin  |Kopi    |4                |NULL       |Tersedia|
|Nasi Goreng Kampung|Makanan |NULL             |2          |Habis   |
|Mie Goreng Pedas   |Makanan |NULL             |3          |Tersedia|
|Roti Bakar Coklat  |Makanan |NULL             |0          |Tersedia|
|Kentang Goreng     |Makanan |NULL             |1          |Tersedia|
|Pisang Goreng      

In [13]:
from pyspark.sql.functions import avg, count, round as sround

df_menu.groupBy("kategori").agg(
    sround(avg("harga"), 2).alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).orderBy("kategori").show()

spark.stop()
print("SparkSession ditutup.")

+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
|    Kopi|        22000.0|          7|
| Makanan|        20000.0|          5|
|Non-Kopi|        15000.0|          2|
+--------+---------------+-----------+

SparkSession ditutup.


Tantangan utama saat mengonversi data MongoDB menjadi Spark DataFrame adalah perbedaan filosofi skema. MongoDB bersifat schema-on-read: setiap dokumen boleh memiliki field berbeda, sehingga tingkat_kepahitan hanya ada pada menu Kopi dan pedas_level hanya pada menu Makanan. Spark DataFrame bersifat tabular dengan skema seragam, jadi field yang tidak dimiliki sebuah dokumen harus direpresentasikan sebagai null. Masalah lain muncul pada tipe data: kolom _id bertipe ObjectId tidak dikenali Spark, dan jika data melewati pandas, kolom bilangan bulat yang memiliki nilai kosong otomatis menjadi float dengan NaN sehingga gagal diterima oleh IntegerType. Jika Spark menebak skema sendiri, ia juga bisa salah menebak atau gagal ketika kolom hanya berisi nilai kosong.

Saya mengatasinya dengan tiga langkah. Pertama, _id diubah menjadi string. Kedua, setiap dokumen dibaca dengan dict.get() sehingga field yang tidak ada otomatis bernilai None. Ketiga, skema StructType didefinisikan secara eksplisit dengan nullable=True untuk field opsional. Hasilnya, konversi stabil dan agregasi seperti groupBy tetap benar, karena fungsi seperti avg mengabaikan nilai null